# Milestone 4: Threshold Selection, Fair Lending, and Adverse Action

This notebook applies the final production model (constrained + calibrated LightGBM from Milestone 3) to three credit risk decision problems:

1. **Threshold selection** (Section 4A) -- choosing the approve/decline cutoff under asymmetric costs
2. **Fair lending analysis** (Section 4B) -- measuring disparate impact at the chosen threshold
3. **Adverse action explainability** (Section 4C) -- generating regulator-compliant denial reasons via SHAP

Sections 4B and 4C live in separate notebooks (`05_fair_lending.ipynb`, `06_shap_adverse_action.ipynb`).

## Section 4A: Cost-Sensitive Threshold Selection

A credit risk model produces a probability of default. The business decision -- approve or decline -- requires a threshold. There is no single "correct" threshold; the answer depends on the relative cost of two errors:

- **False Negative (FN)**: approved a borrower who defaults. Loss ≈ outstanding principal × LGD. For a typical Home Credit cash loan: roughly 280k currency units (500k principal × 70% LGD × 80% timing factor).
- **False Positive (FP)**: declined a borrower who would have paid. Loss ≈ foregone net profit. Roughly 80k currency units.

The FN:FP ratio for non-prime consumer lending typically sits in the **3:1 to 8:1** range. We compute the optimal threshold across this range rather than committing to a single ratio.

The threshold analysis uses the validation set (not test). Test is held back for final reporting.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

# Load predictions saved at the end of Milestone 3
with open(Path("../data/processed/predictions.pkl"), "rb") as f:
    preds = pickle.load(f)

y_val = preds["y_val"]
y_val_pred = preds["y_val_pred_final"]  # constrained + calibrated LightGBM

print(f"Validation set: {len(y_val):,} applicants, {y_val.mean():.2%} default rate")
print(f"Mean predicted PD: {y_val_pred.mean():.4f}")

In [ ]:
# Cost units (currency unit per applicant); ratio is what matters
COST_FP = 80_000  # foregone profit if you decline someone who would have paid
# FN cost will vary with the ratio we're testing


def threshold_costs(y_true, y_pred, threshold, fn_cost, fp_cost):
    """
    Apply the threshold and return cost components.
    Threshold rule: PD > threshold = decline (predicted positive class).
    """
    declined = y_pred > threshold

    # FN: approved (declined=False) and actually defaulted (y_true=1)
    fn_count = ((~declined) & (y_true == 1)).sum()
    # FP: declined (declined=True) and would have paid (y_true=0)
    fp_count = (declined & (y_true == 0)).sum()
    # Total approval rate
    approval_rate = (~declined).mean()
    # Default rate among approved
    if (~declined).sum() > 0:
        post_threshold_default_rate = y_true[~declined].mean()
    else:
        post_threshold_default_rate = 0.0

    total_cost = fn_count * fn_cost + fp_count * fp_cost

    return {
        "threshold": threshold,
        "fn_count": fn_count,
        "fp_count": fp_count,
        "total_cost": total_cost,
        "approval_rate": approval_rate,
        "post_default_rate": post_threshold_default_rate,
    }


# Sweep thresholds across a dense range
thresholds = np.linspace(0.01, 0.50, 100)

# Run for multiple cost ratios
ratios = [2, 3.5, 5, 8, 10]
results_by_ratio = {}

for ratio in ratios:
    fn_cost = COST_FP * ratio
    rows = [threshold_costs(y_val, y_val_pred, t, fn_cost, COST_FP) for t in thresholds]
    df_r = pd.DataFrame(rows)
    df_r["ratio"] = ratio
    results_by_ratio[ratio] = df_r

# Combine
all_results = pd.concat(results_by_ratio.values(), ignore_index=True)
print(f"Computed cost at {len(thresholds)} thresholds × {len(ratios)} ratios = {len(all_results):,} configurations")

In [ ]:
# For each ratio, find the threshold that minimizes total cost
optimal_by_ratio = []
for ratio, df_r in results_by_ratio.items():
    best_row = df_r.loc[df_r["total_cost"].idxmin()].copy()
    optimal_by_ratio.append({
        "ratio": ratio,
        "fn_cost": COST_FP * ratio,
        "fp_cost": COST_FP,
        "optimal_threshold": best_row["threshold"],
        "approval_rate": best_row["approval_rate"],
        "post_default_rate": best_row["post_default_rate"],
        "fn_count": int(best_row["fn_count"]),
        "fp_count": int(best_row["fp_count"]),
        "total_cost_units": best_row["total_cost"],
    })

optimal_df = pd.DataFrame(optimal_by_ratio)
print("Optimal threshold by cost ratio:")
print(optimal_df[["ratio", "optimal_threshold", "approval_rate", "post_default_rate", "fn_count", "fp_count"]].round(4).to_string(index=False))

In [ ]:
# Three-panel plot:
# Left: total cost vs threshold for each ratio
# Middle: approval rate vs threshold
# Right: post-threshold default rate vs threshold

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: total cost
for ratio in ratios:
    df_r = results_by_ratio[ratio]
    axes[0].plot(df_r["threshold"], df_r["total_cost"] / 1e9, label=f"FN:FP = {ratio}:1")
    # Mark optimal
    opt = df_r.loc[df_r["total_cost"].idxmin()]
    axes[0].scatter(opt["threshold"], opt["total_cost"] / 1e9, s=80, zorder=5)
axes[0].set_xlabel("Decline threshold (PD)")
axes[0].set_ylabel("Total cost (billions of currency units)")
axes[0].set_title("Total cost curve by ratio")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Panel 2: approval rate (same across ratios -- only depends on threshold)
axes[1].plot(thresholds, results_by_ratio[3.5]["approval_rate"], color="steelblue")
for ratio in ratios:
    df_r = results_by_ratio[ratio]
    opt = df_r.loc[df_r["total_cost"].idxmin()]
    axes[1].axvline(opt["threshold"], color="gray", linestyle="--", alpha=0.4)
    axes[1].scatter(opt["threshold"], opt["approval_rate"], s=80, zorder=5,
                    label=f"{ratio}:1 → {opt['approval_rate']:.1%}")
axes[1].set_xlabel("Decline threshold (PD)")
axes[1].set_ylabel("Approval rate")
axes[1].set_title("Approval rate at each optimal threshold")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

# Panel 3: post-threshold default rate among approved
axes[2].plot(thresholds, results_by_ratio[3.5]["post_default_rate"], color="indianred")
axes[2].axhline(y_val.mean(), color="black", linestyle="--", alpha=0.5,
                label=f"Population rate {y_val.mean():.2%}")
for ratio in ratios:
    df_r = results_by_ratio[ratio]
    opt = df_r.loc[df_r["total_cost"].idxmin()]
    axes[2].scatter(opt["threshold"], opt["post_default_rate"], s=80, zorder=5)
axes[2].set_xlabel("Decline threshold (PD)")
axes[2].set_ylabel("Default rate among approved")
axes[2].set_title("Default rate post-threshold")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# A different way to see the analysis: how the optimal threshold moves as the
# FN:FP ratio changes. Compact view of the sensitivity of the operating point
# to cost assumptions.

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(optimal_df["ratio"], optimal_df["optimal_threshold"], marker="o", color="steelblue")
ax.set_xlabel("FN:FP cost ratio")
ax.set_ylabel("Optimal decline threshold")
ax.set_title("How the optimal threshold shifts with cost assumptions")
ax.grid(alpha=0.3)

# Annotate each point
for _, row in optimal_df.iterrows():
    ax.annotate(
        f"τ={row['optimal_threshold']:.3f}\napproval={row['approval_rate']:.1%}",
        xy=(row["ratio"], row["optimal_threshold"]),
        xytext=(8, -8),
        textcoords="offset points",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

In [ ]:
# Save the chosen threshold for use in 4B and 4C notebooks
chosen_ratio = 5.0
chosen_threshold = optimal_df.loc[optimal_df["ratio"] == chosen_ratio, "optimal_threshold"].iloc[0]
secondary_threshold = optimal_df.loc[optimal_df["ratio"] == 8.0, "optimal_threshold"].iloc[0]

threshold_metadata = {
    "chosen_ratio": chosen_ratio,
    "chosen_threshold": float(chosen_threshold),
    "chosen_approval_rate": float(optimal_df.loc[optimal_df["ratio"] == chosen_ratio, "approval_rate"].iloc[0]),
    "secondary_threshold_8to1": float(secondary_threshold),
    "cost_fp": COST_FP,
    "cost_fn_at_5to1": COST_FP * chosen_ratio,
    "optimal_table": optimal_df.to_dict("records"),
}

with open("../data/processed/threshold_metadata.pkl", "wb") as f:
    pickle.dump(threshold_metadata, f)

print(f"Saved threshold metadata for downstream milestones.")
print(f"  Primary operating point: FN:FP = {chosen_ratio}:1, threshold = {chosen_threshold:.4f}")
print(f"  Secondary (for sensitivity): FN:FP = 8:1, threshold = {secondary_threshold:.4f}")

**Section 4A takeaways:**

| FN:FP ratio | Optimal threshold | Approval rate | Default rate among approved |
|---|---|---|---|
| 2:1 | 0.327 | 98.4% | 7.46% |
| 3.5:1 | 0.203 | 94.4% | 6.64% |
| 5:1 | 0.168 | 87.7% | 5.68% |
| 8:1 | 0.109 | 75.8% | 4.47% |
| 10:1 | 0.089 | 67.3% | 3.80% |

**Key observations:**

- As the FN:FP ratio increases, the optimal threshold drops and the approval rate drops. This makes sense: when FNs are relatively more costly, the business should decline more aggressively to avoid them, accepting more FPs (declined-but-would-have-paid) in exchange.
- At the low end (2:1), the model does almost nothing -- 98.4% approval rate means only 1.6% of applicants are declined. The cost asymmetry is too weak to justify aggressive screening. No real lender would operate here, but the finding is instructive: cost optimization alone doesn't always produce a defensible threshold.
- At the high end (10:1), approval drops to 67.3% and default rate among approved falls to 3.80%. This reflects a very risk-averse lender at the subprime end of the market. Fair lending disparity will likely spike at this threshold.
- Across the defensible range for non-prime cash lending (3.5:1 to 8:1), the model consistently produces default rates among approved of 4.5-6.6%, meaningfully lower than the 8.07% population rate. The model is doing real work.

**Pragmatic operating point for downstream analysis:** FN:FP = 5:1, threshold = 0.168, approval rate = 87.7%.

Reasoning for the choice:
- 5:1 sits at a defensible midpoint for non-prime consumer lending -- reasonable given typical LGDs of 70% for unsecured cash loans and moderate profit margins per loan.
- 87.7% approval rate reflects "meaningful screening without being punitive" -- declining ~12% of applicants and reducing portfolio default rate from 8.07% to 5.68%.
- Enables sensitivity analysis: fair lending metrics in 4B are also evaluated at 8:1 (more conservative) to show how disparity shifts as the threshold moves.

**Caveat on real-world threshold selection:** In production, cost optimization is a starting point, not the final answer. Real thresholds are also constrained by portfolio loss rate ceilings, volume targets, fair lending compliance, and operational capacity. The threshold I chose here reflects the cost-optimization frame; Milestone 4B will assess whether it also passes fair lending scrutiny.

This threshold (0.168) will be used in Section 4B to evaluate approval rate disparity across gender and age.